In [2]:
import pandas as pd
import numpy as np

In [14]:
data = pd.read_excel("C:/Users/detto/Downloads/orbis_dataset.xlsx")
data.shape

(5506, 256)

In [15]:
data = data.drop(columns=[col for col in data.columns if col.endswith(('.1', '.2', '.3', '.4', '.5'))])
data = data.replace(["n.d.", "NaN", "nan", "NAN", " na", " NA", "n.a.", ""], np.nan)
data = data.drop(data.columns[[-3, -4]], axis=1)
data = data.drop(columns=[col for col in data.columns if col.endswith("Year - 2")])
data = data.drop(columns=['#', 'Inactive', 'OwnData', 'Country ISO code', 'BvD sectors', 'Status'])
data['NACE Rev. 2, core code (4 digits)'] = data['NACE Rev. 2, core code (4 digits)'].astype(str).str[:2]

# Extract base names
cols_last = [col for col in data.columns if col.endswith("Last avail. yr")]
cols_year_1 = [col for col in data.columns if col.endswith("Year - 1")]

# Base name extraction helper
def get_base(col, suffix):
    return col.replace(suffix, "").strip()

# Build maps from base name to full column name
base_to_last = {get_base(col, "Last avail. yr"): col for col in cols_last}
base_to_year_1 = {get_base(col, "Year - 1"): col for col in cols_year_1}

# Ensure numeric
for col in cols_year_1 + cols_last:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Compute % delta
for base in base_to_last.keys() & base_to_year_1.keys():
    col_last = base_to_last[base]
    col_year_1 = base_to_year_1[base]
    delta_col = f"{base} Δ%"
    data[delta_col] = (data[col_last] - data[col_year_1]) / data[col_year_1]

# Drop original "Year - 1" columns
cols_to_drop = list(base_to_year_1.values())
data.drop(columns=cols_to_drop, inplace=True)


C:\Users\detto\AppData\Local\Temp\ipykernel_29048\3193637979.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace(["n.d.", "NaN", "nan", "NAN", " na", " NA", "n.a.", ""], np.nan)
C:\Users\detto\AppData\Local\Temp\ipykernel_29048\3193637979.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[delta_col] = (data[col_last] - data[col_year_1]) / data[col_year_1]
C:\Users\detto\AppData\Local\Temp\ipykernel_29048\3193637979.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of

In [16]:
threshold = 0.6
data = data.loc[:, data.isna().mean() <= threshold]

In [17]:
data.shape

(5506, 62)